<a href="https://colab.research.google.com/github/Islamintech/AI/blob/main/AI_Assignment2_CNN_20252403_%EC%9D%B4%EC%8A%AC%EB%A1%AC%EB%B2%A1_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets


!pip install -q datasets
from datasets import load_dataset

hf_train = load_dataset('uoft-cs/cifar10', split='train')
hf_test = load_dataset('uoft-cs/cifar10', split='test')

class HFCifar10(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.ds = hf_dataset
        self.transform = transform
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item['img'].convert('RGB')
        label = item['label']
        if self.transform:
            img = self.transform(img)
        return img, label

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_datset = HFCifar10(hf_train, transform=transform_train)
test_dataset = HFCifar10(hf_test, transform=transform_test)


class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()
    torch.manual_seed(2026)
    self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(32)
    self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(32)

    self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
    self.bn3 = nn.BatchNorm2d(64)
    self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
    self.bn4 = nn.BatchNorm2d(64)

    self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
    self.bn5 = nn.BatchNorm2d(128)
    self.conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
    self.bn6 = nn.BatchNorm2d(128)

    self.pool = nn.MaxPool2d(2, 2)
    self.relu = nn.ReLU()
    self.dropout_conv = nn.Dropout(0.25)
    self.dropout_fc = nn.Dropout(0.5)

    self.fc1 = nn.Linear(128 * 4 * 4, 256)
    self.fc2 = nn.Linear(256, 10)

  def forward(self, x):
    x = self.relu(self.bn1(self.conv1(x)))
    x = self.relu(self.bn2(self.conv2(x)))
    x = self.pool(x)            # 32x32 -> 16x16
    x = self.dropout_conv(x)

    x = self.relu(self.bn3(self.conv3(x)))
    x = self.relu(self.bn4(self.conv4(x)))
    x = self.pool(x)            # 16x16 -> 8x8
    x = self.dropout_conv(x)

    x = self.relu(self.bn5(self.conv5(x)))
    x = self.relu(self.bn6(self.conv6(x)))
    x = self.pool(x)            # 8x8 -> 4x4
    x = self.dropout_conv(x)

    x = x.view(x.size(0), -1)
    x = self.relu(self.fc1(x))
    x = self.dropout_fc(x)
    x = self.fc2(x)
    return x

model = CNN().cuda()


train_batchsize = 128
train_loader = torch.utils.data.DataLoader(dataset=train_datset, batch_size=train_batchsize, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
num_epochs = 30

model.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.cuda(), labels.cuda()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pred = outputs.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    print(f'Epoch [{epoch+1}/{num_epochs}]  Loss: {running_loss/len(train_loader):.4f}  Train Acc: {100*correct/total:.2f}%')



batchsize = 64
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                           batch_size=batchsize, shuffle=True)

num_correct = 0

model.eval()

with torch.no_grad():
  for image, label in test_loader:
    image = image.cuda()
    label = label.cuda()
    output = model(image)
    pred = output.argmax(dim=1)
    num_correct += (pred == label).sum()

print(f'Accuracy : {num_correct / len(test_dataset) * 100:.2f} %')


README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 23.9MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Epoch [1/30]  Loss: 1.6446  Train Acc: 38.57%
Epoch [2/30]  Loss: 1.2595  Train Acc: 55.03%
Epoch [3/30]  Loss: 1.0968  Train Acc: 61.20%
Epoch [4/30]  Loss: 0.9915  Train Acc: 65.30%
Epoch [5/30]  Loss: 0.9255  Train Acc: 67.96%
Epoch [6/30]  Loss: 0.8677  Train Acc: 70.15%
Epoch [7/30]  Loss: 0.8235  Train Acc: 72.04%
Epoch [8/30]  Loss: 0.7777  Train Acc: 73.76%
Epoch [9/30]  Loss: 0.7347  Train Acc: 75.27%
Epoch [10/30]  Loss: 0.7066  Train Acc: 76.23%
Epoch [11/30]  Loss: 0.6778  Train Acc: 77.57%
Epoch [12/30]  Loss: 0.6558  Train Acc: 78.20%
Epoch [13/30]  Loss: 0.6266  Train Acc: 79.23%
Epoch [14/30]  Loss: 0.6104  Train Acc: 79.94%
Epoch [15/30]  Loss: 0.5844  Train Acc: 80.55%
Epoch [16/30]  Loss: 0.5516  Train Acc: 81.71%
Epoch [17/30]  Loss: 0.5348  Train Acc: 82.33%
Epoch [18/30]  Loss: 0.5194  Train Acc: 82.79%
Epoch [19/30]  Loss: 0.5002  Train Acc: 83.39%
Epoch [20/30]  Loss: 0.4834  Train Acc: 83.94%
Epoch [21/30]  Loss: 0.4730  Train Acc: 84.31%
Epoch [22/30]  Loss: 0